# 15.6 Testing in Practice — Coverage, Properties, Structure and CI

**Prerequisites:** 15.1–15.5, 07 Module and Packages, 14.16 Interview Patterns  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- Where tests live: `src/` layout, `tests/`, and configuration in `pyproject.toml`
- **Coverage** — what it measures, and 🔴 exactly what 100% does *not* prove
- Statement vs **branch** coverage, measured on the same file
- **Property-based testing** — `14.16`'s `verify()` harness, grown up, with `hypothesis`
- `doctest` — examples in your docstrings, executed
- Flaky tests, slow tests, and finding both
- A CI pipeline that would actually catch things
- Interview questions

---

## Where tests live

Two layouts dominate. The difference matters more than it looks.

```
   FLAT LAYOUT                        SRC LAYOUT  (recommended)
   project/                           project/
   ├── jobs/                          ├── src/
   │   ├── __init__.py                │   └── jobs/
   │   └── retry.py                   │       ├── __init__.py
   ├── tests/                         │       └── retry.py
   │   └── test_retry.py              ├── tests/
   └── pyproject.toml                 │   └── test_retry.py
                                      └── pyproject.toml
```

🔴 **The flat layout tests the wrong thing.** Because `jobs/` sits in the working directory,
`import jobs` finds it whether or not the package is correctly installed. Your tests pass; a
user who `pip install`s your package gets `ModuleNotFoundError` because you forgot to include a
subpackage.

With `src/`, the only way to import `jobs` is to **install it** (`pip install -e .`), so your
tests exercise the same import path your users will. Packaging proper is **17**.

### Configuration

Put it in `pyproject.toml` and have one file for everything:

```toml
[tool.pytest.ini_options]
testpaths = ["tests"]
addopts = "-ra --strict-markers --strict-config"
filterwarnings = ["error"]
markers = [
    "slow: takes more than a second; excluded from the pre-commit run",
    "integration: needs a real database or network",
]

[tool.coverage.run]
branch = true
source = ["src"]

[tool.coverage.report]
fail_under = 80
show_missing = true
exclude_also = ["if TYPE_CHECKING:", "raise NotImplementedError"]
```

Four of those lines are load-bearing:

| Setting | Why |
|---|---|
| `--strict-markers` | a typo'd mark becomes an error, not a silent no-op (**15.3**) |
| `--strict-config` | a typo'd *config key* becomes an error too |
| `filterwarnings = ["error"]` | deprecations fail the build while they are still cheap to fix |
| `branch = true` | statement coverage overstates how much you have tested — proved below |

In [ ]:
import shutil
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

WORK = Path(tempfile.mkdtemp(prefix="py156_"))


def make_project(files, name="proj"):
    project = Path(tempfile.mkdtemp(prefix=f"{name}_", dir=WORK))
    for relpath, source in files.items():
        path = project / relpath
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(textwrap.dedent(source).lstrip("\n"), encoding="utf-8")
    return project


def run_in(project, *args, module="pytest"):
    """Run `python -m <module> ...` inside `project` and return its output."""
    done = subprocess.run([sys.executable, "-m", module, *args], cwd=project,
                          capture_output=True, text=True, encoding="utf-8",
                          errors="replace", timeout=300)
    return (f"$ python -m {module} {' '.join(args)}".rstrip() + "\n" + "-" * 70 + "\n"
            + (done.stdout + done.stderr).rstrip()
            + "\n" + "-" * 70 + f"\nexit code: {done.returncode}")


def have(package):
    import importlib.util
    return importlib.util.find_spec(package) is not None


print("scratch    :", WORK)
print("coverage   :", "installed" if have("coverage") else "NOT installed")
print("hypothesis :", "installed" if have("hypothesis") else "NOT installed")

## Coverage — and what it does not tell you

**Coverage** measures which lines of your code ran while the tests ran. That is all it measures.
It is a *lower bound* on how much you have tested, and people routinely read it as an upper
bound on how many bugs remain.

The file below has **two real bugs**. The test suite reaches **100% statement coverage**.

In [ ]:
RETRYING = {
    "retrying.py": """
        RETRYABLE = (500, 502, 503)


        def is_retryable(status, attempts, max_attempts=3):
            \"\"\"Should a failed request be retried?\"\"\"
            if status in RETRYABLE:
                return attempts < max_attempts
            return False


        def apply_cap(delay, cap):
            \"\"\"Clamp a retry delay to a ceiling.\"\"\"
            if delay > cap:
                delay = cap
            return delay
    """,
    "test_retrying.py": r"""
        from retrying import is_retryable, apply_cap


        def test_server_error_is_retryable():
            assert is_retryable(500, attempts=0) is True


        def test_budget_is_respected():
            assert is_retryable(500, attempts=3) is False


        def test_client_error_is_not_retryable():
            assert is_retryable(404, attempts=0) is False


        def test_cap_clamps_a_large_delay():
            assert apply_cap(120.0, cap=30.0) == 30.0
    """,
}

cov = make_project(RETRYING, name="cov")
print(run_in(cov, "run", "--source=.", "--omit=test_*", "-m", "pytest", "-q",
             "-p", "no:cacheprovider", module="coverage"))
print()
print(run_in(cov, "report", "-m", module="coverage"))

**100%.** Four passing tests, every line executed. Now the two bugs.

In [ ]:
import importlib
import sys as _sys

_sys.path.insert(0, str(cov))
retrying = importlib.import_module("retrying")

print("BUG 1 - a missing case:")
print("  is_retryable(429, 0) =", retrying.is_retryable(429, 0))
print("  429 is 'Too Many Requests'. It is the single most retryable status there is,")
print("  and it is absent from RETRYABLE. No line is missing, so coverage sees nothing.")

print("\nBUG 2 - an untested branch:")
print("  apply_cap(5.0, cap=30.0) =", retrying.apply_cap(5.0, cap=30.0))
print("  Correct - but no test ever took the `delay <= cap` path. If the function")
print("  returned `cap` unconditionally, all four tests would still pass.")

_sys.path.remove(str(cov))
del _sys.modules["retrying"]

🔴 **Bug 1 is the important one: coverage is structurally incapable of finding
missing code.** It measures lines that exist. A requirement you never implemented has no line
to miss.

Bug 2, though, coverage *can* find — if you ask it to measure **branches** rather than
statements. Branch coverage asks not "did this line run?" but "did this `if` go both ways?".

In [ ]:
cov_branch = make_project(RETRYING, name="covbranch")
print(run_in(cov_branch, "run", "--branch", "--source=.", "--omit=test_*",
             "-m", "pytest", "-q", "-p", "no:cacheprovider", module="coverage"))
print()
print(run_in(cov_branch, "report", "-m", module="coverage"))

Same code, same tests, and the number drops from **100% to 92%** with
`BrPart 1` — one **partial** branch. The `Missing` column names it: `13->15`, meaning "the jump
from line 13 to line 15 never happened" — the `delay <= cap` case.

🔴 **Always set `branch = true`.** Statement coverage flatters you.

### How to read a coverage number

| Reading | Verdict |
|---|---|
| "we are at 100%, we are done" | wrong — see bug 1 above |
| "this module is at 12%, nobody has ever tested it" | **useful** — that is the real value |
| "coverage dropped 4% in this PR" | **useful** — new code arrived untested |
| "raise the target to 95%" | usually counter-productive — it buys tests written to touch lines |

Use coverage to **find the untested corners**, not to score yourself. `fail_under` is worth
setting as a ratchet that stops things getting worse, not as a goal.

## Property-based testing

In **14.16** you wrote `verify(fast, slow, generate)`: generate random inputs, compare a fast
implementation against a slow one you trust. That is **property-based testing**, and it is the
highest-yield technique in this whole folder.

The shift in thinking:

| Example-based (everything so far) | Property-based |
|---|---|
| "`truncate('hello world', 8)` is `'hello...'`" | "**for all** text and limits, the result is never longer than the limit" |
| you pick the inputs | the library picks thousands, including the nasty ones |
| finds bugs you thought of | finds bugs you did not |

`hypothesis` generates inputs, and when it finds a failure it **shrinks** it — repeatedly
simplifying the counterexample until it is minimal. That last part is what makes it usable: it
hands you `text='0', limit=0` rather than a 400-character Unicode string.

### Properties worth looking for

| Property | Shape |
|---|---|
| **Round trip** | `decode(encode(x)) == x` — the most valuable one |
| **Invariant** | `len(result) <= limit`, `sorted(x)` is a permutation of `x` |
| **Idempotence** | `f(f(x)) == f(x)` — normalising, cleaning, formatting |
| **Oracle** | agrees with a slow-but-obviously-correct version — **14.16**'s `verify` |
| **Never crashes** | `f` accepts any input of this type without raising |

In [ ]:
props = make_project({
    "text_utils.py": """
        def truncate(text, limit):
            \"\"\"Shorten `text` to at most `limit` characters, with an ellipsis.\"\"\"
            if len(text) > limit:
                return text[:limit - 3] + "..."
            return text
    """,
    "test_props.py": r"""
        from hypothesis import given, settings, strategies as st

        from text_utils import truncate


        @given(st.text(), st.integers(min_value=0, max_value=50))
        @settings(max_examples=500, derandomize=True)
        def test_never_longer_than_the_limit(text, limit):
            assert len(truncate(text, limit)) <= limit
    """,
}, name="props")

if have("hypothesis"):
    print(run_in(props, "-q", "-p", "no:cacheprovider", "--tb=long"))
else:
    print("hypothesis is not installed - `pip install hypothesis` to run this cell.")
    print("The exercises below include a stdlib-only version of the same idea.")

Look at what it handed back:

```
text = '0', limit = 0
E   AssertionError: assert 3 <= 0
E    +  where 3 = len('...')
E    +    where '...' = truncate('0', 0)
```

The **minimal** counterexample. The bug: when `limit < 3`, `text[:limit - 3]` slices with a
*negative* index, and then three dots are added regardless — so `truncate` can return something
**longer** than the limit it was given. No example-based test anyone writes by hand starts with
`limit=0`.

Hypothesis also prints an *Explanation* naming the line only executed by failing cases. And it
records failures, so the counterexample is retried first on every subsequent run.

> 🔴 **`derandomize=True` is used here only so this notebook prints the same thing every time.**
> In a real suite leave it off — you *want* fresh inputs each run. Use `@example(...)` to pin
> specific cases you care about permanently.

### Without `hypothesis`

You do not need the library to get most of the benefit — **14.16**'s harness is twenty lines
and works with the standard library alone. The cell below is that harness, applied to the same
bug.

In [ ]:
import random
import string


def check_property(prop, generate, trials=2000, seed=1156):
    """Run `prop(case)` on random inputs; return the first failing case.

    The stdlib version of hypothesis. No shrinking - so it will hand you an
    ugly counterexample rather than a minimal one, which is exactly the
    feature you are paying `hypothesis` for.
    """
    rng = random.Random(seed)
    for _ in range(trials):
        case = generate(rng)
        try:
            if not prop(case):
                return case
        except Exception as exc:                       # a crash is a failure too
            return (case, f"{type(exc).__name__}: {exc}")
    return None


def truncate(text, limit):
    if len(text) > limit:
        return text[:limit - 3] + "..."
    return text


def generate_case(rng):
    length = rng.randint(0, 12)
    text = "".join(rng.choice(string.ascii_letters + " .") for _ in range(length))
    return text, rng.randint(0, 15)


counterexample = check_property(
    prop=lambda case: len(truncate(*case)) <= case[1],
    generate=generate_case,
)
print("first failing case:", counterexample)

text, limit = counterexample
print(f"  truncate({text!r}, {limit}) -> {truncate(text, limit)!r}")
print(f"  length {len(truncate(text, limit))} > limit {limit}")

# The fix, and the same property re-checked.
def truncate_fixed(text, limit):
    if len(text) <= limit:
        return text
    if limit <= 3:
        return text[:limit]              # no room for an ellipsis
    return text[:limit - 3] + "..."


still_failing = check_property(
    prop=lambda case: len(truncate_fixed(*case)) <= case[1],
    generate=generate_case,
)
print("\nafter the fix, failing case:", still_failing, "(None = 2000 cases passed)")

Notice the difference in the two counterexamples. `hypothesis` shrank to
`('0', 0)`; the hand-rolled version returned whatever it happened to hit first. Both find the
bug — shrinking is what makes it *cheap to understand*.

## `doctest` — examples that cannot go stale

An example in a docstring is documentation. Run it, and it is also a test.

```bash
python -m doctest fmt.py -v      # standalone
pytest --doctest-modules         # as part of the suite
```

🔴 Doctests are **exact string comparisons** on the `repr`. They are excellent for small pure
functions with stable output, and terrible for dicts before 3.7, floats, addresses, tracebacks
and anything with a timestamp.

In [ ]:
doct = make_project({
    "fmt.py": """
        def humanise(seconds):
            \"\"\"Format a duration in seconds.

            >>> humanise(90)
            '1m30s'
            >>> humanise(0)
            '0s'
            >>> humanise(45)
            '45sec'
            \"\"\"
            if seconds == 0:
                return "0s"
            minutes, secs = divmod(seconds, 60)
            return f"{minutes}m{secs}s" if minutes else f"{secs}s"
    """,
}, name="doct")

print(run_in(doct, "--doctest-modules", "-q", "-p", "no:cacheprovider"))

The docstring **claimed** `humanise(45)` returns `'45sec'`. It returns
`'45s'`. Documentation that lies, caught automatically — which is the entire argument for
doctests.

## Flaky and slow tests

A **flaky** test passes and fails on the same code. It is worse than a failing test, because it
trains everyone to re-run CI instead of reading it.

| Cause | Fix |
|---|---|
| Real time (`datetime.now`, `sleep`) | inject a clock (**15.5**) |
| Unseeded `random` | seed it, or make the property hold for all inputs |
| Test order / shared state | function-scoped fixtures (**15.4**) |
| Dict/set **iteration order** across processes | sort before comparing, or use `assertCountEqual` |
| Real network | fake it (**15.5**); keep a few real integration tests |
| Concurrency (**12**) | the hardest; make the test deterministic, do not add `sleep` |

Find them by running the suite repeatedly and in a different order — `pytest -p randomly` or
simply `pytest --lf` after a shuffle. Then **quarantine or fix**; never just re-run.

For slow tests, `--durations` is the whole tool:

```bash
pytest --durations=10          # the ten slowest
pytest -m "not slow"           # the fast subset, for pre-commit
pytest -n auto                 # parallel, with pytest-xdist
```

In [ ]:
timing = make_project({
    "test_timing.py": r"""
        import time

        import pytest


        def test_fast_one():
            assert 2 ** 10 == 1024


        @pytest.mark.slow
        def test_a_slow_one():
            time.sleep(0.30)
            assert True


        @pytest.mark.slow
        def test_another_slow_one():
            time.sleep(0.15)
            assert True
    """,
    "pytest.ini": r"""
        [pytest]
        markers =
            slow: takes more than a second; excluded from the pre-commit run
    """,
}, name="timing")

print(run_in(timing, "--durations=3", "-q", "-p", "no:cacheprovider", "--strict-markers"))
print()
print("The fast subset a pre-commit hook would run:")
print(run_in(timing, "-m", "not slow", "-q", "-p", "no:cacheprovider", "--strict-markers"))

## A CI pipeline worth having

This is not run here — it needs a CI service — but it is the shape to copy. Every line in it
exists because of something earlier in this folder.

```yaml
# .github/workflows/tests.yml
name: tests
on: [push, pull_request]

jobs:
  test:
    runs-on: ubuntu-latest
    strategy:
      fail-fast: false
      matrix:
        python-version: ["3.12", "3.13", "3.14"]
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: ${{ matrix.python-version }}
      - run: pip install -e ".[dev]"
      - run: ruff check .                     # lint            (18)
      - run: mypy src                         # type check      (16)
      - run: |                                # the tests
          coverage run -m pytest -ra --strict-markers --strict-config
          coverage report --fail-under=80
```

| Line | Why it is there |
|---|---|
| `fail-fast: false` | you want *all* versions' results, not just the first failure |
| the version matrix | your users are not all on your version |
| `pip install -e .` | tests the installed package, not the working directory |
| `--strict-markers --strict-config` | typos fail the build (**15.3**) |
| `coverage report --fail-under` | the ratchet — stops coverage sliding |

🔴 **Do not put `-x` in CI.** You want the complete list of failures from one run, not one per
push. And treat **exit code 5** (no tests collected, **15.3**) as a failure, not a pass.

## Putting the folder together

A short decision table for what to reach for.

| Situation | Tool | Notebook |
|---|---|---|
| Check one specific input/output | plain `assert` in a test function | **15.1**, **15.3** |
| Many inputs, same logic | `@pytest.mark.parametrize` | **15.3** |
| A rule that should hold for *all* inputs | `hypothesis` | **15.6** |
| Two implementations, one trusted | oracle test — `verify()` | **14.16**, **15.6** |
| Setup shared by several tests | a fixture | **15.4** |
| Needs a temp directory | `tmp_path` | **15.4** |
| Depends on the clock, network, or a DB | inject it; fake it | **15.5** |
| A known bug you are not fixing today | `xfail(strict=True)` | **15.3** |
| An example in a docstring | `--doctest-modules` | **15.6** |
| Finding untested corners | `coverage --branch` | **15.6** |

And the order to write them in: **make it fail, make it pass, make it clean.** A test written
after the fix has never been seen to fail, so you have no evidence it would.

## Interview Questions

1. **What is the difference between a unit, integration and end-to-end test?** What does the
   test pyramid actually claim, and when is it wrong? *(**15.1**)*
2. **Why should `assert` never be used to validate user input?** *(`python -O` removes it —
   **15.1**)*
3. **What is the difference between a failure and an error** in `unittest`, and why does
   `pytest` not distinguish them? *(**15.2**)*
4. **How does `pytest` make `assert a == b` print a diff**, when the statement carries no such
   information? *(bytecode rewriting at import — **15.3**)*
5. **`subTest` vs `@parametrize`** — what does each generate, and how many tests does the
   report count? *(**15.2**, **15.3**)*
6. **What is a fixture's scope**, and what is the risk of `scope="session"`? *(**15.4**)*
7. **A colleague's mock "isn't working" — the real function still runs.** What do you ask
   first? *(where are they patching — **15.5**)*
8. **Stub vs mock.** Why does a suite full of mocks get harder to refactor? *(**15.5**)*
9. **Your code has 100% coverage. What can you still be confident is untested?** *(missing
   requirements; untaken branches under statement coverage — **15.6**)*
10. **What is a flaky test, and why is it worse than a failing one?** *(**15.6**)*
11. **Give three properties worth testing** for a function that serialises and deserialises a
    record. *(round trip, invariant, never-crashes — **15.6**)*
12. **You inherit a 40,000-line codebase with no tests. Where do you start?** *(coverage to
    find the untested core; characterisation tests around what you must change first; never a
    coverage target)*
13. **When would you deliberately not write a test?** *(throwaway scripts, a spike, generated
    code, a test that only restates the implementation)*

In [ ]:
# ---- tidy up ----
shutil.rmtree(WORK, ignore_errors=True)
print("scratch removed:", not WORK.exists())

---

## Common Mistakes & Pitfalls

1. 🔴 **Reading 100% coverage as 'fully tested'.** Coverage cannot see code you never wrote — a missing requirement has no line to miss.
2. 🔴 **Statement coverage without `branch = true`.** It reports 100% on a file with an `if` whose false path no test ever takes.
3. **Chasing a coverage target.** It produces tests that execute lines without asserting anything useful.
4. **Using the flat layout.** `import jobs` works from the project root whether or not the package is correctly installed, so packaging bugs reach your users.
5. **Re-running CI until a flaky test passes.** That trains the team to ignore red.
6. **Doctests on floats, dicts of unstable order, addresses or timestamps.** They compare `repr` strings exactly.
7. **`derandomize=True` left on in a real hypothesis suite.** You lose the fresh inputs that made it worth using.
8. **`-x` in CI.** One failure fixed per push instead of a full list.
9. **Treating pytest exit code 5 as success.** Zero tests collected is a broken pipeline.
10. **Writing the test after the fix.** A test that has never failed is not known to work.

## Best Practices

- Use the `src/` layout and `pip install -e .`, so tests import the way users will.
- Keep all configuration in `pyproject.toml`, with `--strict-markers --strict-config` and `filterwarnings = ["error"]`.
- Turn on `branch = true`, and use `fail_under` as a ratchet rather than a goal.
- Use coverage to find untested modules, not to score the team.
- Add one property-based test for anything with a round trip, an invariant, or a trusted slow version to compare against.
- Mark slow and integration tests so a fast subset can run on every save.
- Run the suite in a randomised order periodically — order dependence is silent until it is not.
- Fix flakes or quarantine them explicitly; never leave them re-running.
- Write the failing test first. Watch it fail. Then fix it.

## Practice Exercises

Try these before moving on.

1. 🔴 Add `429` to `RETRYABLE` and a test for it. Did the coverage number move? What does that tell you about coverage as a quality measure?
2. Turn on `branch = true` for a project of your own and find the first partial branch. Was it a real gap?
3. Write a round-trip property for a `to_json` / `from_json` pair using `hypothesis`, and let it find at least one input you would never have chosen.
4. Take `truncate_fixed` from this notebook and check a *second* property: that the result is always a prefix of the original when `limit <= 3`. Does it hold?
5. Re-implement `check_property` with a crude shrinker: on failure, retry with smaller inputs until it stops failing. Compare its output with hypothesis's.
6. Add doctests to three functions you wrote in **04 Functions**, then run `pytest --doctest-modules`. How many were already wrong?
7. 🔴 Write a deliberately flaky test (use `datetime.now()`), run it 100 times with `pytest --count`, and then fix it by injecting a clock.
8. Write the CI workflow above for one of your own projects and make it fail for the right reason at least once.
9. **Capstone:** take any notebook from **14 Data Structure and Algorithm**, extract its implementations into a module, and build a real test suite — parametrised examples, an oracle test against brute force, a property-based test, and branch coverage over 90%.

---

## Version notes

| Version | Change |
|---|---|
| **coverage 7** | `exclude_also` (used above) supplements rather than replaces the default `exclude_lines`; `--fail-under` accepts decimals |
| **hypothesis 6** | `@settings(derandomize=True)`, the failure database, and the `Explanation` output shown here |
| **pytest 8** | `--strict-config` and `--strict-markers` recommended as defaults |
| **Python 3.12** | `sys.monitoring` — coverage tools use it for markedly lower overhead |
| **Python 3.11** | `tomllib` in the stdlib, so `pyproject.toml` needs no third-party parser |

Written against **pytest 9.1.1**, **coverage 7.15.4**, **hypothesis 6.165.10** on **Python 3.14.4**.

## 15 Testing and Debugging — the folder

| Notebook | Covers |
|---|---|
| **15.1** | why test; `assert`; `python -O`; AAA; a hand-rolled runner; the pyramid |
| **15.2** | `unittest` — `TestCase`, lifecycle, `subTest`, skips, discovery |
| **15.3** | `pytest` — assertion rewriting, `approx`, `raises`, `parametrize`, marks, the CLI |
| **15.4** | fixtures — scopes, `yield`, `conftest.py`, `tmp_path`, `monkeypatch` |
| **15.5** | test doubles — `Mock`, where to patch, `autospec`, fakes, injection |
| **15.6** | this notebook — layout, coverage, properties, doctest, flakiness, CI |
| **15.7** | tracebacks, chained exceptions, `excepthook`, logging, `faulthandler` |
| **15.8** | `pdb`, `breakpoint()`, post-mortem, `pytest --pdb` |
| **15.9** | debugging method, bisection, delta debugging, heisenbugs |
| **15.10** | logging — configuration, handlers, structured output |

## Where next — the debugging half

Everything so far tells you *that* something is wrong. **15.7–15.9** are about finding out
**why**, and they close the loop: the last step of debugging is writing the test that would
have caught it.

| Notebook | Covers |
|---|---|
| **15.7 Reading Failures** | tracebacks in depth, chained exceptions, logging, hung processes |
| **15.8 The Interactive Debugger** | `pdb`, `breakpoint()`, post-mortem, `pytest --pdb` |
| **15.9 Debugging in Practice** | the method, `git bisect`, delta debugging, and bugs that move |

## Related

- **14.16 Interview Patterns** — the `verify()` harness that becomes property-based testing here
- **07 Module and Packages** — imports, which decide whether `src/` or flat works
- **16 Type Hints and Static Typing** — `mypy`, the other half of the CI pipeline above
- **17 Tooling, Packaging and Environments** — `pyproject.toml`, `ruff`, `pip install -e .`
- **19 Capstone Projects** — where the whole of this folder gets used at once